# 가맹점 업종 분류 데모

이 노트북은 `store_classifier_demo.zip`을 업로드한 뒤 샘플 데이터 생성, 모델 학습, 테스트, 추론을 실행합니다.

## 1. 프로젝트 ZIP 업로드

In [1]:
from google.colab import files as colab_files

uploaded = colab_files.upload()
zip_candidates = [name for name in uploaded if name.lower().endswith('.zip')]

if not zip_candidates:
    raise ValueError('store_classifier_demo.zip 파일을 업로드해 주세요.')

zip_name = zip_candidates[0]
print('업로드 완료:', zip_name)

Saving store_classifier_demo.zip to store_classifier_demo.zip
업로드 완료: store_classifier_demo.zip


## 2. 압축 해제 및 작업 폴더 이동

In [2]:
import os
import shutil
import zipfile

project_dir = '/content/store_classifier_demo'

if os.path.exists(project_dir):
    shutil.rmtree(project_dir)

with zipfile.ZipFile(zip_name, 'r') as zip_file:
    zip_file.extractall('/content')

if not os.path.exists(os.path.join(project_dir, 'main.py')):
    raise FileNotFoundError('압축 파일 안에서 store_classifier_demo/main.py를 찾지 못했습니다.')

os.chdir(project_dir)
print('현재 폴더:', os.getcwd())
print('\n프로젝트 파일:')
print('\n'.join(sorted(os.listdir('.'))))

현재 폴더: /content/store_classifier_demo

프로젝트 파일:
README.md
data
data_preprocess.py
log
logger.py
main.py
meta_data
model
model.py
requirements.txt
result
sample_data.py
tools.py
utils.py


## 3. 라이브러리 설치

In [3]:
!pip -q install -r requirements.txt

## 4. 실행 장치 확인

In [4]:
import torch

device = 'gpu' if torch.cuda.is_available() else 'cpu'
print('PyTorch:', torch.__version__)
print('CUDA 사용 가능:', torch.cuda.is_available())
print('main.py 실행 장치:', device)

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA 사용 가능: True
main.py 실행 장치: gpu
GPU: Tesla T4


## 5. 문법 검사

In [5]:
!python -m py_compile main.py model.py data_preprocess.py utils.py tools.py logger.py sample_data.py
print('문법 검사 완료')

문법 검사 완료


## 6. 전체 데모 실행

샘플 데이터 생성 → 학습 → 테스트 → 추론 순서로 실행됩니다.

In [6]:
!python main.py --mode demo --epochs 3 --batch-size 8 --device {device}

2026-07-21 13:49:08 | INFO | ##############################
2026-07-21 13:49:08 | INFO | mode=demo, device=cuda
샘플 데이터 생성 완료
2026-07-21 13:49:08 | INFO | 학습 샘플: 50
2026-07-21 13:49:08 | INFO | 검증 샘플: 10
Epoch: [1][0/7]	Time  1.368 ( 1.368)	Data  0.010 ( 0.010)	Loss 1.6310e+00 (1.6310e+00)	Acc@1  12.50 ( 12.50)	Acc@5 100.00 (100.00)
Epoch: [1][6/7]	Time  0.091 ( 0.215)	Data  0.000 ( 0.002)	Loss 1.5194e+00 (1.5833e+00)	Acc@1   0.00 ( 22.00)	Acc@5 100.00 (100.00)
Test: [1/2]	Time  0.002 ( 0.002)	Loss 1.5353e+00 (1.5974e+00)	Acc@1   0.00 ( 20.00)	Acc@5 100.00 (100.00)
checkpoint 저장: ./model/model_20260721_134915_checkpoint.pt
2026-07-21 13:49:15 | INFO | epoch=1 train_loss=1.5833 train_acc1=22.00 train_acc5=100.00 val_loss=1.5974 val_acc1=20.00 val_acc5=100.00
Epoch: [2][0/7]	Time  0.005 ( 0.005)	Data  0.001 ( 0.001)	Loss 1.4236e+00 (1.4236e+00)	Acc@1  50.00 ( 50.00)	Acc@5 100.00 (100.00)
Epoch: [2][6/7]	Time  0.005 ( 0.005)	Data  0.000 ( 0.000)	Loss 1.5216e+00 (1.3994e+00)	Acc@1   0.00 ( 

## 7. 생성 결과 확인

In [7]:
from glob import glob
import pandas as pd

result_files = sorted(glob('result/*.csv'))
model_files = sorted(glob('model/*.pt'))

print('모델 파일:')
for path in model_files:
    print(' -', path)

print('\n결과 파일:')
for path in result_files:
    print(' -', path)

for path in result_files:
    print(f'\n[{path}]')
    display(pd.read_csv(path).head(10))

모델 파일:
 - model/model_20260721_134915_checkpoint.pt

결과 파일:
 - result/inference_result_20260721_134915.csv
 - result/test_incorrect_table_20260721_134915.csv
 - result/test_result_20260721_134915.csv

[result/inference_result_20260721_134915.csv]


,id,text,store_category_index,scores,store_category_cd,store_category_nm,store_category_top_nm,store_category_mid_nm,asset_management_nm
0,0,아메리카노 커피 카페,0,0.448143,C000,카페,음식,카페,일반
1,1,김치찌개 한식당,0,0.309620,C000,카페,음식,카페,일반
2,2,야간 편의점 도시락,0,0.406190,C000,카페,음식,카페,일반
3,3,건강 처방전 약국,4,0.453642,C004,의류,소매,의류,일반
4,4,여성 패션 의류 매장,0,0.348660,C000,카페,음식,카페,일반



[result/test_incorrect_table_20260721_134915.csv]


,label,all,correct1,correct5,acc1,acc5,pred_1_1,pred_1_2,pred_1_3,pred_1_4,...,pred_5_1,pred_5_2,pred_5_3,pred_5_4,pred_5_5,rate_5_1,rate_5_2,rate_5_3,rate_5_4,rate_5_5
0,0,2,2,2,1.0,1.0,0.0,NaN,NaN,NaN,...,0,1,2,3,4,1.0,1.0,1.0,1.0,1.0
1,1,2,1,2,0.5,1.0,1.0,4.0,NaN,NaN,...,0,1,2,3,4,1.0,1.0,1.0,1.0,1.0
2,2,2,0,2,0.0,1.0,0.0,NaN,NaN,NaN,...,0,1,2,3,4,1.0,1.0,1.0,1.0,1.0
3,3,2,2,2,1.0,1.0,3.0,NaN,NaN,NaN,...,0,1,2,3,4,1.0,1.0,1.0,1.0,1.0
4,4,2,1,2,0.5,1.0,0.0,4.0,NaN,NaN,...,0,1,2,3,4,1.0,1.0,1.0,1.0,1.0



[result/test_result_20260721_134915.csv]


,id,text,label,pred1,pred2,pred3,pred4,pred5,scores,correct1,correct5
0,0,커피와 디저트 카페,0,0,4,3,1,2,0.464984,1,1
1,1,로스터리 커피숍,0,0,4,1,3,2,0.374700,1,1
2,2,불고기 한식당,1,1,4,0,3,2,0.218988,1,1
3,3,국밥 백반집,1,4,0,1,3,2,0.305322,0,1
4,4,24시간 편의점,2,0,4,1,2,3,0.325352,0,1
5,5,간편식 스토어,2,0,4,1,2,3,0.352372,0,1
6,6,처방 약국,3,3,4,0,1,2,1.220555,1,1
7,7,메디컬 약국,3,3,4,0,1,2,1.468669,1,1
8,8,남성 패션 의류,4,4,0,3,1,2,0.383046,1,1
9,9,캐주얼 옷 매장,4,0,4,1,3,2,0.329743,0,1


## 8. 원하는 문장으로 추론

In [8]:
custom_texts = [
    '아메리카노와 케이크를 파는 카페',
    '김치찌개 백반 전문점',
    '24시간 도시락 편의점',
    '처방전 조제 약국',
    '여성 캐주얼 의류 매장',
]

pd.DataFrame({'text': custom_texts}).to_csv(
    'data/pred_data.csv',
    index=False,
)

display(pd.read_csv('data/pred_data.csv'))

,text
0,아메리카노와 케이크를 파는 카페
1,김치찌개 백반 전문점
2,24시간 도시락 편의점
3,처방전 조제 약국
4,여성 캐주얼 의류 매장


In [9]:
!python main.py --mode inference --result inference --device {device}

2026-07-21 13:49:36 | INFO | ##############################
2026-07-21 13:49:36 | INFO | mode=inference, device=cuda
Test: [0/1]	Time  0.322 ( 0.322)	Loss 1.3317e+00 (1.3317e+00)	Acc@1  80.00 ( 80.00)	Acc@5 100.00 (100.00)
2026-07-21 13:49:36 | INFO | checkpoint: ./model/model_20260721_134915_checkpoint.pt
2026-07-21 13:49:36 | INFO | loss=1.3317, acc1=80.00, acc5=100.00
2026-07-21 13:49:36 | INFO | 결과 저장: ./result/inference_result_20260721_134936.csv
 id              text  store_category_index   scores store_category_cd store_category_nm store_category_top_nm store_category_mid_nm asset_management_nm
  0 아메리카노와 케이크를 파는 카페                     0 0.413983              C000                카페                    음식                    카페                  일반
  1       김치찌개 백반 전문점                     0 0.397263              C000                카페                    음식                    카페                  일반
  2      24시간 도시락 편의점                     0 0.425950              C000               

In [10]:
inference_files = sorted(glob('result/inference_result_*.csv'))
latest_inference = inference_files[-1]
print('최신 추론 결과:', latest_inference)
display(pd.read_csv(latest_inference))

최신 추론 결과: result/inference_result_20260721_134936.csv


,id,text,store_category_index,scores,store_category_cd,store_category_nm,store_category_top_nm,store_category_mid_nm,asset_management_nm
0,0,아메리카노와 케이크를 파는 카페,0,0.413983,C000,카페,음식,카페,일반
1,1,김치찌개 백반 전문점,0,0.397263,C000,카페,음식,카페,일반
2,2,24시간 도시락 편의점,0,0.425950,C000,카페,음식,카페,일반
3,3,처방전 조제 약국,4,0.498544,C004,의류,소매,의류,일반
4,4,여성 캐주얼 의류 매장,0,0.404919,C000,카페,음식,카페,일반


## 9. 결과 CSV 다운로드

In [ ]:
colab_files.download(latest_inference)